# Comparacao entre janela real, pose `vitpose+motionbert` e saida Kimodo

Este notebook:

- resolve uma `prompt_id` window-level nos manifests reais do projeto
- carrega a `window.mp4` correspondente
- renderiza a pose 3D extraida pelo pipeline `vitpose+motionbert`
- renderiza a saida `motion_amass.npz` do Kimodo via SMPL-X
- monta um video comparativo lado a lado

A estrutura foi inspirada em `smplx_video_notebook.ipynb`, mas aqui o foco e comparar a mesma janela real com a pose extraida e a geracao do Kimodo.

In [1]:
# Instale as dependencias se necessario.
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "smplx",
        "torch",
        "numpy",
        "trimesh",
        "pyrender",
        "imageio",
        "imageio-ffmpeg",
        "matplotlib",
        "pillow",
        "ipython",
        "PyOpenGL",
        "PyOpenGL_accelerate",
    ]
)

0

In [2]:
from __future__ import annotations

from pathlib import Path
import json
import os
import textwrap

PYOPENGL_PLATFORM = os.environ.get("PYOPENGL_PLATFORM", "egl")
os.environ["PYOPENGL_PLATFORM"] = PYOPENGL_PLATFORM

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Markdown, Video, display

import torch
import smplx
import trimesh
import pyrender

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["axes.grid"] = False

print(f"PYOPENGL_PLATFORM={PYOPENGL_PLATFORM}")

PYOPENGL_PLATFORM=egl


In [3]:
ROOT_DIR = Path.cwd()
WINDOW_MANIFEST_PATH = ROOT_DIR / "output" / "robot_emotions_qwen_windows" / "window_description_manifest.jsonl"
ANCHOR_CATALOG_PATH = ROOT_DIR / "output" / "robot_emotions_kimodo_anchors" / "kimodo_anchor_catalog.jsonl"
GENERATION_MANIFEST_PATH = ROOT_DIR / "output" / "robot_emotions_kimodo_generated" / "kimodo_generation_manifest.jsonl"
MODEL_PATH = ROOT_DIR / "models_lockedhead"

PROMPT_ID = "robot_emotions_10ms_u02_tag11__w000"
# PROMPT_ID = None  # Use a primeira entrada do manifest de geracao.

OUT_DIR = ROOT_DIR / "output" / "robot_emotions_kimodo_generated" / "notebook_renders"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FPS = 30.0
PANEL_WIDTH = 480
PANEL_HEIGHT = 480
LABEL_HEIGHT = 38
PANEL_RESAMPLE = Image.Resampling.BICUBIC

POSE_VIEW_ELEV = 18
POSE_VIEW_AZIM = -72
POSE_LINE_WIDTH = 3.0
POSE_POINT_SIZE = 36
POSE_CENTER_ON_ROOT = False
POSE_FLIP_X = False
POSE_SHOW_TRAIL = True
POSE_TRAIL_SECONDS = 1.2
POSE_DPI = 96
POSE_FIGSIZE = (PANEL_WIDTH / POSE_DPI, PANEL_HEIGHT / POSE_DPI)

SMPLX_DEFAULT_GENDER = "neutral"
SMPLX_BG_COLOR = [1.0, 1.0, 1.0, 1.0]
SMPLX_AMBIENT_LIGHT = [0.35, 0.35, 0.35]
SMPLX_DIRECTIONAL_LIGHT_INTENSITY = 2.0
SMPLX_CAM_X = 0.0
SMPLX_CAM_Y = 1.0
SMPLX_CAM_Z = 1.5
SMPLX_ROT_X = np.pi / 3
SMPLX_ROT_Y = 0.0
SMPLX_ROT_Z = np.pi

In [4]:
def load_json(path: str | Path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_jsonl(path: str | Path):
    entries = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                entries.append(json.loads(line))
    return entries


def format_path(path: str | Path) -> str:
    path = Path(path)
    try:
        return str(path.relative_to(ROOT_DIR))
    except ValueError:
        return str(path)


generation_entries = load_jsonl(GENERATION_MANIFEST_PATH)
window_entries = load_jsonl(WINDOW_MANIFEST_PATH)
anchor_entries = load_jsonl(ANCHOR_CATALOG_PATH)

generation_by_prompt = {entry["prompt_id"]: entry for entry in generation_entries}
window_by_prompt = {entry["prompt_id"]: entry for entry in window_entries}
anchor_by_prompt = {entry["prompt_id"]: entry for entry in anchor_entries}

if PROMPT_ID is None:
    PROMPT_ID = generation_entries[0]["prompt_id"]

if PROMPT_ID not in generation_by_prompt:
    raise KeyError(f"PROMPT_ID={PROMPT_ID!r} nao encontrado no manifest de geracao.")
if PROMPT_ID not in window_by_prompt:
    raise KeyError(f"PROMPT_ID={PROMPT_ID!r} nao encontrado no manifest de janelas.")
if PROMPT_ID not in anchor_by_prompt:
    raise KeyError(f"PROMPT_ID={PROMPT_ID!r} nao encontrado no catalogo ancorado.")

SELECTED_GENERATION = generation_by_prompt[PROMPT_ID]
SELECTED_WINDOW = window_by_prompt[PROMPT_ID]
SELECTED_ANCHOR = anchor_by_prompt[PROMPT_ID]

OUTPUT_STEM = PROMPT_ID
POSE_VIDEO_PATH = OUT_DIR / f"{OUTPUT_STEM}__motionbert_pose.mp4"
KIMODO_VIDEO_PATH = OUT_DIR / f"{OUTPUT_STEM}__kimodo_smplx.mp4"
COMPARISON_VIDEO_PATH = OUT_DIR / f"{OUTPUT_STEM}__comparison.mp4"

WINDOW_VIDEO_PATH = Path(SELECTED_WINDOW["artifacts"]["window_video_path"])
POSE3D_NPZ_PATH = Path(SELECTED_WINDOW["source"]["pose3d_npz_path"])
TRACEABILITY_PATH = Path(SELECTED_ANCHOR["constraints_path"]).with_name("traceability.json")
KIMODO_AMASS_PATH = Path(SELECTED_GENERATION["artifacts"]["amass_npz_path"])

overview = {
    "prompt_id": PROMPT_ID,
    "reference_clip_id": SELECTED_GENERATION["reference_clip_id"],
    "window_start_sec": SELECTED_WINDOW["window"]["start_sec"],
    "window_end_sec": SELECTED_WINDOW["window"]["end_sec"],
    "window_duration_sec": SELECTED_WINDOW["window"]["duration_sec"],
    "window_video_path": format_path(WINDOW_VIDEO_PATH),
    "pose3d_npz_path": format_path(POSE3D_NPZ_PATH),
    "traceability_path": format_path(TRACEABILITY_PATH),
    "kimodo_amass_path": format_path(KIMODO_AMASS_PATH),
    "labels": SELECTED_GENERATION.get("labels", {}),
}

print("Janela selecionada:")
print(json.dumps(overview, indent=2, ensure_ascii=False))
print()

prompt_text = SELECTED_GENERATION.get("prompt_text", "")
if prompt_text:
    print("Prompt do Kimodo:")
    print(textwrap.fill(prompt_text, width=100))

Janela selecionada:
{
  "prompt_id": "robot_emotions_10ms_u02_tag11__w000",
  "reference_clip_id": "robot_emotions_10ms_u02_tag11",
  "window_start_sec": 0.0,
  "window_end_sec": 10.0,
  "window_duration_sec": 10.0,
  "window_video_path": "output/robot_emotions_qwen_windows/robot_emotions_10ms_u02_tag11__w000/window.mp4",
  "pose3d_npz_path": "output/robot_emotions_pose3d/10ms/user_02/robot_emotions_10ms_u02_tag11/pose/pose3d/pose3d.npz",
  "traceability_path": "output/robot_emotions_kimodo_anchors/robot_emotions_10ms_u02_tag11__w000/traceability.json",
  "kimodo_amass_path": "output/robot_emotions_kimodo_generated/robot_emotions_10ms_u02_tag11__w000/motion_amass.npz",
  "labels": {
    "emotion": "Happiness",
    "action": null,
    "stimulus": "Autobiographical recall"
  }
}

Prompt do Kimodo:
A person stands still with arms clasped and head steady, making small hand gestures while
maintaining upright posture


In [5]:
DEFAULT_IMUGPT22_PARENT_INDICES = (
    -1,
    0,
    0,
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    9,
    9,
    12,
    13,
    14,
    16,
    17,
    18,
    19,
)


def npz_scalar(value):
    if isinstance(value, np.ndarray) and value.shape == ():
        return value.item()
    return value


def load_pose_sequence(path: str | Path) -> dict[str, object]:
    payload = np.load(path, allow_pickle=True)
    return {
        "path": Path(path),
        "clip_id": str(npz_scalar(payload["clip_id"])),
        "joint_names_3d": [str(name) for name in payload["joint_names_3d"].tolist()],
        "skeleton_parents": payload["skeleton_parents"].astype(np.int32).tolist()
        if "skeleton_parents" in payload
        else list(DEFAULT_IMUGPT22_PARENT_INDICES),
        "joint_positions_xyz": payload["joint_positions_xyz"].astype(np.float32),
        "joint_confidence": payload["joint_confidence"].astype(np.float32)
        if "joint_confidence" in payload
        else np.ones(payload["joint_positions_xyz"].shape[:2], dtype=np.float32),
        "frame_indices": payload["frame_indices"].astype(np.int32)
        if "frame_indices" in payload
        else np.arange(payload["joint_positions_xyz"].shape[0], dtype=np.int32),
        "timestamps_sec": payload["timestamps_sec"].astype(np.float32)
        if "timestamps_sec" in payload
        else None,
        "fps": float(npz_scalar(payload["fps"])) if "fps" in payload else 20.0,
        "coordinate_space": str(npz_scalar(payload["coordinate_space"]))
        if "coordinate_space" in payload
        else "unknown",
    }


FULL_POSE = load_pose_sequence(POSE3D_NPZ_PATH)
TRACEABILITY = load_json(TRACEABILITY_PATH)
SOURCE_KEYFRAMES = np.asarray(TRACEABILITY["source_keyframes"], dtype=np.int32)

if SOURCE_KEYFRAMES.ndim != 1 or len(SOURCE_KEYFRAMES) == 0:
    raise ValueError("source_keyframes vazio ou invalido em traceability.json")
if SOURCE_KEYFRAMES.max() >= FULL_POSE["joint_positions_xyz"].shape[0]:
    raise IndexError("traceability.json referencia frames fora do pose3d.npz")

POSE_SEQUENCE = {
    "clip_id": FULL_POSE["clip_id"],
    "joint_names_3d": FULL_POSE["joint_names_3d"],
    "skeleton_parents": FULL_POSE["skeleton_parents"],
    "joint_positions_xyz": FULL_POSE["joint_positions_xyz"][SOURCE_KEYFRAMES],
    "joint_confidence": FULL_POSE["joint_confidence"][SOURCE_KEYFRAMES],
    "frame_indices": SOURCE_KEYFRAMES,
    "timestamps_sec": SOURCE_KEYFRAMES.astype(np.float32) / max(FULL_POSE["fps"], 1e-6)
    if FULL_POSE["timestamps_sec"] is None
    else FULL_POSE["timestamps_sec"][SOURCE_KEYFRAMES],
    "fps": float(TARGET_FPS),
    "coordinate_space": FULL_POSE["coordinate_space"],
}

pose_overview = {
    "pose_clip_id": POSE_SEQUENCE["clip_id"],
    "pose_coordinate_space": POSE_SEQUENCE["coordinate_space"],
    "pose_num_frames_aligned": int(POSE_SEQUENCE["joint_positions_xyz"].shape[0]),
    "pose_num_joints": int(POSE_SEQUENCE["joint_positions_xyz"].shape[1]),
    "source_pose_fps": FULL_POSE["fps"],
    "target_fps": TARGET_FPS,
    "source_frame_start": int(TRACEABILITY["source_frame_start"]),
    "source_frame_end": int(TRACEABILITY["source_frame_end"]),
}

print("Resumo da pose alinhada para a janela:")
print(json.dumps(pose_overview, indent=2, ensure_ascii=False))

Resumo da pose alinhada para a janela:
{
  "pose_clip_id": "robot_emotions_10ms_u02_tag11",
  "pose_coordinate_space": "pseudo_global_metric",
  "pose_num_frames_aligned": 300,
  "pose_num_joints": 22,
  "source_pose_fps": 20.0,
  "target_fps": 30.0,
  "source_frame_start": 0,
  "source_frame_end": 298
}


In [6]:
def apply_pose_render_transforms(points: np.ndarray) -> np.ndarray:
    transformed = np.array(points, copy=True)
    if POSE_CENTER_ON_ROOT:
        transformed -= transformed[:, :1, :]
    if POSE_FLIP_X:
        transformed[..., 0] *= -1.0
    return transformed


def bone_color(joint_name: str) -> str:
    name = joint_name.lower()
    if "left" in name:
        return "#2f6db3"
    if "right" in name:
        return "#e58f2a"
    return "#2f855a"


def compute_scene_limits(points: np.ndarray):
    flat = points.reshape(-1, 3)
    mins = np.nanmin(flat, axis=0)
    maxs = np.nanmax(flat, axis=0)
    center = (mins + maxs) / 2.0
    span = max(float(np.max(maxs - mins)), 0.8)
    half = span * 0.6
    return tuple((float(value - half), float(value + half)) for value in center)


def set_axes_equal(ax, limits):
    (x_min, x_max), (y_min, y_max), (z_min, z_max) = limits
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_zlim(z_min, z_max)
    if hasattr(ax, "set_box_aspect"):
        ax.set_box_aspect((x_max - x_min, y_max - y_min, z_max - z_min))


def draw_pose_frame(
    ax,
    frame_points: np.ndarray,
    parents: list[int],
    joint_names: list[str],
    *,
    limits,
    title: str,
    trail_points: np.ndarray | None = None,
):
    ax.cla()
    ax.set_facecolor("white")

    if trail_points is not None and len(trail_points) > 1:
        ax.plot(
            trail_points[:, 0],
            trail_points[:, 1],
            trail_points[:, 2],
            color="#222222",
            linewidth=1.5,
            alpha=0.5,
        )

    for child_index, parent_index in enumerate(parents):
        if parent_index < 0:
            continue
        segment = frame_points[[parent_index, child_index], :]
        ax.plot(
            segment[:, 0],
            segment[:, 1],
            segment[:, 2],
            color=bone_color(joint_names[child_index]),
            linewidth=POSE_LINE_WIDTH,
            alpha=0.95,
        )

    ax.scatter(
        frame_points[:, 0],
        frame_points[:, 1],
        frame_points[:, 2],
        c=[bone_color(name) for name in joint_names],
        s=POSE_POINT_SIZE,
        depthshade=False,
    )

    set_axes_equal(ax, limits)
    ax.set_title(title, fontsize=10)
    ax.view_init(elev=POSE_VIEW_ELEV, azim=POSE_VIEW_AZIM)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_zlabel("")
    ax.grid(False)

    for axis in ("xaxis", "yaxis", "zaxis"):
        pane = getattr(ax, axis).pane
        pane.set_facecolor((1.0, 1.0, 1.0, 0.0))
        pane.set_edgecolor((0.9, 0.9, 0.9, 1.0))


def figure_to_rgb_array(fig) -> np.ndarray:
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba(), dtype=np.uint8)
    return np.array(rgba[..., :3], copy=True)


def render_pose_video(output_path: str | Path) -> Path:
    output_path = Path(output_path)
    points = apply_pose_render_transforms(POSE_SEQUENCE["joint_positions_xyz"])
    limits = compute_scene_limits(points)
    root_points = points[:, 0, :]
    trail_window = max(2, int(round(POSE_TRAIL_SECONDS * TARGET_FPS)))

    fig = plt.figure(figsize=POSE_FIGSIZE, dpi=POSE_DPI)
    ax = fig.add_subplot(111, projection="3d")

    with imageio.get_writer(
        output_path,
        fps=float(TARGET_FPS),
        codec="libx264",
        quality=7,
        ffmpeg_log_level="error",
    ) as writer:
        for frame_index, frame_points in enumerate(points):
            trail_points = None
            if POSE_SHOW_TRAIL and not POSE_CENTER_ON_ROOT:
                trail_start = max(0, frame_index - trail_window + 1)
                trail_points = root_points[trail_start : frame_index + 1]

            draw_pose_frame(
                ax,
                frame_points,
                POSE_SEQUENCE["skeleton_parents"],
                POSE_SEQUENCE["joint_names_3d"],
                limits=limits,
                title=(
                    f"Pose vitpose+motionbert | frame {frame_index + 1}/{len(points)} | "
                    f"t={POSE_SEQUENCE['timestamps_sec'][frame_index]:.2f}s"
                ),
                trail_points=trail_points,
            )
            writer.append_data(figure_to_rgb_array(fig))

    plt.close(fig)
    return output_path


pose_rendered_path = render_pose_video(POSE_VIDEO_PATH)
print(f"Video da pose salvo em: {format_path(pose_rendered_path)}")
display(Video(str(pose_rendered_path), embed=True))

Video da pose salvo em: output/robot_emotions_kimodo_generated/notebook_renders/robot_emotions_10ms_u02_tag11__w000__motionbert_pose.mp4


In [7]:
def _as_float32(array) -> np.ndarray:
    return np.asarray(array, dtype=np.float32)


def _fit_feature_dim(arr, target_dim: int) -> np.ndarray:
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr[None]
    current_dim = int(arr.shape[-1])
    if current_dim == target_dim:
        return arr
    if current_dim > target_dim:
        return arr[..., :target_dim]
    pad_width = [(0, 0)] * arr.ndim
    pad_width[-1] = (0, target_dim - current_dim)
    return np.pad(arr, pad_width, mode="constant")


def normalize_amass_payload(path: str | Path):
    data = np.load(path, allow_pickle=True)
    keys = set(data.files)

    if not {"pose_body", "root_orient"}.issubset(keys):
        raise KeyError(
            "O arquivo Kimodo precisa conter `pose_body` e `root_orient` para a renderizacao SMPL-X."
        )

    pose_payload = {
        "body_pose": _as_float32(data["pose_body"]),
        "global_orient": _as_float32(data["root_orient"]),
    }
    if "trans" in data:
        pose_payload["transl"] = _as_float32(data["trans"])
    if "betas" in data:
        pose_payload["betas"] = _as_float32(data["betas"])
    if "pose_jaw" in data:
        pose_payload["jaw_pose"] = _as_float32(data["pose_jaw"])
    if "pose_eye" in data:
        pose_eye = _as_float32(data["pose_eye"])
        pose_payload["leye_pose"] = pose_eye[:, :3]
        pose_payload["reye_pose"] = pose_eye[:, 3:]
    if "pose_hand" in data:
        pose_hand = _as_float32(data["pose_hand"])
        pose_payload["left_hand_pose"] = pose_hand[:, :45]
        pose_payload["right_hand_pose"] = pose_hand[:, 45:]

    gender = str(npz_scalar(data["gender"])) if "gender" in data else SMPLX_DEFAULT_GENDER
    fps = float(npz_scalar(data["mocap_frame_rate"])) if "mocap_frame_rate" in data else TARGET_FPS

    return data, pose_payload, gender, fps


def get_optional_tensor(pose_data: dict[str, np.ndarray], name: str, frame_index: int, total_frames: int):
    if name not in pose_data:
        return None

    arr = pose_data[name]
    if arr.ndim == 1:
        arr = arr[None]
    elif arr.shape[0] == total_frames:
        arr = arr[frame_index : frame_index + 1]
    else:
        arr = arr[:1]
    return torch.tensor(arr, dtype=torch.float32)


def get_betas(model, pose_data: dict[str, np.ndarray], frame_index: int, total_frames: int):
    target_dim = int(getattr(model, "num_betas", 10))
    if "betas" not in pose_data:
        arr = np.zeros((1, target_dim), dtype=np.float32)
    else:
        arr = _fit_feature_dim(pose_data["betas"], target_dim)
        if arr.ndim == 1:
            arr = arr[None]
        elif arr.shape[0] == total_frames:
            arr = arr[frame_index : frame_index + 1]
        else:
            arr = arr[:1]
    return torch.tensor(arr, dtype=torch.float32)


def get_expression(model, pose_data: dict[str, np.ndarray], frame_index: int, total_frames: int):
    target_dim = int(getattr(model, "num_expression_coeffs", 10))
    if "expression" not in pose_data:
        arr = np.zeros((1, target_dim), dtype=np.float32)
    else:
        arr = _fit_feature_dim(pose_data["expression"], target_dim)
        if arr.ndim == 1:
            arr = arr[None]
        elif arr.shape[0] == total_frames:
            arr = arr[frame_index : frame_index + 1]
        else:
            arr = arr[:1]
    return torch.tensor(arr, dtype=torch.float32)


def create_offscreen_renderer(width: int, height: int):
    try:
        return pyrender.OffscreenRenderer(viewport_width=width, viewport_height=height)
    except Exception as exc:
        raise RuntimeError(
            "Falha ao criar o renderer offscreen. Reinicie o kernel e execute novamente com "
            "PYOPENGL_PLATFORM=egl."
        ) from exc


def build_smplx_scene():
    scene = pyrender.Scene(bg_color=SMPLX_BG_COLOR, ambient_light=SMPLX_AMBIENT_LIGHT)

    camera = pyrender.PerspectiveCamera(yfov=np.pi / 2.0)
    cam_pose = np.array(
        [
            [1, 0, 0, SMPLX_CAM_X],
            [0, 1, 0, SMPLX_CAM_Y],
            [0, 0, 1, SMPLX_CAM_Z],
            [0, 0, 0, 1],
        ],
        dtype=np.float32,
    )

    rotation_x = np.array(
        [
            [1, 0, 0, 0],
            [0, np.cos(SMPLX_ROT_X), -np.sin(SMPLX_ROT_X), 0],
            [0, np.sin(SMPLX_ROT_X), np.cos(SMPLX_ROT_X), 0],
            [0, 0, 0, 1],
        ],
        dtype=np.float32,
    )
    rotation_y = np.array(
        [
            [np.cos(SMPLX_ROT_Y), 0, np.sin(SMPLX_ROT_Y), 0],
            [0, 1, 0, 0],
            [-np.sin(SMPLX_ROT_Y), 0, np.cos(SMPLX_ROT_Y), 0],
            [0, 0, 0, 1],
        ],
        dtype=np.float32,
    )
    rotation_z = np.array(
        [
            [np.cos(SMPLX_ROT_Z), -np.sin(SMPLX_ROT_Z), 0, 0],
            [np.sin(SMPLX_ROT_Z), np.cos(SMPLX_ROT_Z), 0, 0],
            [0, 0, 1, 0],
            [0, 0, 0, 1],
        ],
        dtype=np.float32,
    )

    final_cam = cam_pose @ rotation_z @ rotation_y @ rotation_x
    scene.add(camera, pose=final_cam)

    light = pyrender.DirectionalLight(
        color=np.ones(3),
        intensity=SMPLX_DIRECTIONAL_LIGHT_INTENSITY,
    )
    scene.add(light, pose=cam_pose)
    return scene


def render_kimodo_smplx_video(amass_path: str | Path, output_path: str | Path) -> Path:
    _, pose_data, gender, _ = normalize_amass_payload(amass_path)
    model = smplx.create(
        model_path=str(MODEL_PATH),
        model_type="smplx",
        gender=gender or SMPLX_DEFAULT_GENDER,
        use_pca=False,
        batch_size=1,
    )

    scene = build_smplx_scene()
    renderer = create_offscreen_renderer(PANEL_WIDTH, PANEL_HEIGHT)
    output_path = Path(output_path)

    mesh_node = None
    total_frames = int(pose_data["body_pose"].shape[0])
    with imageio.get_writer(
        output_path,
        fps=float(TARGET_FPS),
        codec="libx264",
        quality=7,
        ffmpeg_log_level="error",
    ) as writer:
        for frame_index in range(total_frames):
            output = model(
                betas=get_betas(model, pose_data, frame_index, total_frames),
                body_pose=torch.tensor(
                    pose_data["body_pose"][frame_index : frame_index + 1],
                    dtype=torch.float32,
                ),
                global_orient=torch.tensor(
                    pose_data["global_orient"][frame_index : frame_index + 1],
                    dtype=torch.float32,
                ),
                transl=get_optional_tensor(pose_data, "transl", frame_index, total_frames),
                left_hand_pose=get_optional_tensor(pose_data, "left_hand_pose", frame_index, total_frames),
                right_hand_pose=get_optional_tensor(pose_data, "right_hand_pose", frame_index, total_frames),
                jaw_pose=get_optional_tensor(pose_data, "jaw_pose", frame_index, total_frames),
                leye_pose=get_optional_tensor(pose_data, "leye_pose", frame_index, total_frames),
                reye_pose=get_optional_tensor(pose_data, "reye_pose", frame_index, total_frames),
                expression=get_expression(model, pose_data, frame_index, total_frames),
                return_verts=True,
            )

            verts = output.vertices.detach().cpu().numpy().squeeze()
            tri_mesh = trimesh.Trimesh(vertices=verts, faces=model.faces, process=False)
            render_mesh = pyrender.Mesh.from_trimesh(tri_mesh, smooth=False)

            if mesh_node is not None:
                scene.remove_node(mesh_node)
            mesh_node = scene.add(render_mesh)

            color, _ = renderer.render(scene)
            writer.append_data(color)

    renderer.delete()
    return output_path


kimodo_rendered_path = render_kimodo_smplx_video(KIMODO_AMASS_PATH, KIMODO_VIDEO_PATH)
print(f"Video Kimodo salvo em: {format_path(kimodo_rendered_path)}")
display(Video(str(kimodo_rendered_path), embed=True))

Video Kimodo salvo em: output/robot_emotions_kimodo_generated/notebook_renders/robot_emotions_10ms_u02_tag11__w000__kimodo_smplx.mp4


In [8]:
def ensure_rgb_uint8(frame: np.ndarray) -> np.ndarray:
    frame = np.asarray(frame)
    if frame.ndim == 2:
        frame = np.repeat(frame[..., None], 3, axis=2)
    if frame.ndim == 3 and frame.shape[2] == 4:
        frame = frame[..., :3]
    if frame.dtype != np.uint8:
        frame = np.clip(frame, 0, 255).astype(np.uint8)
    return frame


def resize_frame(frame: np.ndarray, width: int, height: int) -> np.ndarray:
    frame = ensure_rgb_uint8(frame)
    image = Image.fromarray(frame)
    image = image.resize((width, height), resample=PANEL_RESAMPLE)
    return np.asarray(image, dtype=np.uint8)


def labeled_panel(frame: np.ndarray, label: str) -> np.ndarray:
    frame = resize_frame(frame, PANEL_WIDTH, PANEL_HEIGHT)
    canvas = Image.new("RGB", (PANEL_WIDTH, PANEL_HEIGHT + LABEL_HEIGHT), color=(248, 248, 248))
    canvas.paste(Image.fromarray(frame), (0, LABEL_HEIGHT))
    draw = ImageDraw.Draw(canvas)
    font = ImageFont.load_default()
    draw.rectangle((0, 0, PANEL_WIDTH, LABEL_HEIGHT), fill=(248, 248, 248))
    draw.text((12, 12), label, fill=(20, 20, 20), font=font)
    return np.asarray(canvas, dtype=np.uint8)


def build_comparison_video(
    *,
    window_video_path: str | Path,
    pose_video_path: str | Path,
    kimodo_video_path: str | Path,
    output_path: str | Path,
    num_frames: int,
) -> Path:
    readers = {
        "window": imageio.get_reader(str(window_video_path)),
        "pose": imageio.get_reader(str(pose_video_path)),
        "kimodo": imageio.get_reader(str(kimodo_video_path)),
    }
    labels = {
        "window": "Janela real",
        "pose": "Pose vitpose+motionbert",
        "kimodo": "Kimodo SMPL-X",
    }
    last_frames = {}
    output_path = Path(output_path)

    try:
        with imageio.get_writer(
            output_path,
            fps=float(TARGET_FPS),
            codec="libx264",
            quality=7,
            ffmpeg_log_level="error",
        ) as writer:
            for frame_index in range(int(num_frames)):
                panels = []
                for key in ("window", "pose", "kimodo"):
                    try:
                        frame = readers[key].get_data(frame_index)
                        last_frames[key] = frame
                    except Exception:
                        if key not in last_frames:
                            raise
                        frame = last_frames[key]
                    panels.append(labeled_panel(frame, labels[key]))

                combined = np.concatenate(panels, axis=1)
                writer.append_data(combined)
    finally:
        for reader in readers.values():
            reader.close()

    return output_path


comparison_rendered_path = build_comparison_video(
    window_video_path=WINDOW_VIDEO_PATH,
    pose_video_path=pose_rendered_path,
    kimodo_video_path=kimodo_rendered_path,
    output_path=COMPARISON_VIDEO_PATH,
    num_frames=SELECTED_GENERATION["num_frames"],
)

print(f"Video comparativo salvo em: {format_path(comparison_rendered_path)}")

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1440, 518) to (1440, 528) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Video comparativo salvo em: output/robot_emotions_kimodo_generated/notebook_renders/robot_emotions_10ms_u02_tag11__w000__comparison.mp4


In [ ]:
display(Markdown("## Video comparativo"))
display(Video(str(comparison_rendered_path), embed=True))

display(Markdown("## Videos individuais"))
display(Markdown("### Janela real"))
display(Video(str(WINDOW_VIDEO_PATH), embed=True))
display(Markdown("### Pose alinhada do vitpose+motionbert"))
display(Video(str(pose_rendered_path), embed=True))
display(Markdown("### Saida Kimodo renderizada em SMPL-X"))
display(Video(str(kimodo_rendered_path), embed=True))

## Notas

- Troque `PROMPT_ID` para comparar outra janela ja presente nos manifests.
- O alinhamento temporal da pose usa `traceability.json`, entao a pose 3D e reamostrada na mesma malha temporal da geracao Kimodo.
- Se o renderer SMPL-X falhar no ambiente atual, reinicie o kernel e rode novamente com `PYOPENGL_PLATFORM=egl`.
- O video final e um triptico: `janela real | pose vitpose+motionbert | saida Kimodo`.